# Explore your tapping data

A guided walk from raw landmarks to a result. Run the cells in order.

**Haven't collected anything yet?** Run `python analysis/make_example_data.py`
first, then set `DATA = REPO/'data'/'example'` in the next cell. Everything
below works the same on invented data as on real data.


In [ ]:
import sys, pathlib
REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'analysis' else pathlib.Path.cwd()
sys.path.insert(0, str(REPO / 'analysis'))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import metrics as M

DATA = REPO / 'data' / 'raw'        # <- 'example' to use the invented data
sessions = M.load_all(DATA)
print(f'{len(sessions)} sessions from {DATA}')
for s in sessions:
    print(' ', s['participantId'], '-', len(s['trials']), 'trials')


## 1. What one trial actually looks like

Everything downstream is built on a single number per frame: how far apart
the thumb and index fingertip are, divided by the size of that person's hand.


In [ ]:
session = sessions[0]
trial = session['trials'][0]

t_raw, ratio, metres = M.aperture_signal(trial['frames'])
params = M.TapParams()
t, y = M.resample(t_raw, ratio, params)
y = M.smooth(y, params)
taps = M.detect_taps(t, y, params)

plt.figure(figsize=(13, 4))
plt.plot(t, y, lw=1.4, label='aperture')
plt.plot(taps['tap_times'], y[taps['tap_indices']], 'v', color='crimson',
         ms=8, label=f"taps (n={len(taps['tap_times'])})")
plt.xlabel('time (s)'); plt.ylabel('thumb-index / hand size')
plt.title(f"{session['participantId']} - {trial['id']}")
plt.legend(); plt.grid(alpha=.3); plt.show()


**Check the red triangles.** They should sit at the bottom of each dip, one
per tap. If they don't, the detection settings are wrong for this participant
and every number below is wrong too. Section 4 shows how to change them.


## 2. Every trial, as a table


In [ ]:
rows = []
for s in sessions:
    for tr in s['trials']:
        rows.append({'participant': s['participantId'], **M.analyse_trial(tr, params)})

df = pd.DataFrame(rows)
df[['participant','trialId','hand','n_taps','rate_hz','fft_peak_hz',
    'iti_cv','amplitude_decrement_pct_per_tap','detection_rate']]


### Trials worth checking by hand

Two warning signs: the hand was often not visible, or the counted rate
disagrees with the signal's own frequency (which does not use tap detection
at all, so it is an independent opinion).


In [ ]:
suspect = df[(df.detection_rate < 0.9) |
             ((df.rate_hz - df.fft_peak_hz).abs() > 0.5 * df[['rate_hz','fft_peak_hz']].max(axis=1))]
print(f'{len(suspect)} of {len(df)} trials worth a look')
suspect[['participant','trialId','rate_hz','fft_peak_hz','detection_rate','lost_to_tracking_sec']]


## 3. Left hand versus right

The usual first question. With real participants, use a paired test - each
person contributes both hands.


In [ ]:
if df.hand.notna().any() and df.hand.nunique() > 1:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, col, name in zip(axes,
            ['rate_hz','iti_cv','amplitude_decrement_pct_per_tap'],
            ['tapping rate (Hz)','rhythm variability (CV)','amplitude change (%/tap)']):
        for i, (hand, grp) in enumerate(df.groupby('hand')):
            vals = grp[col].dropna()
            ax.scatter(np.full(len(vals), i) + np.random.default_rng(0).normal(0,.04,len(vals)),
                       vals, s=40, alpha=.7)
            ax.hlines(vals.mean(), i-.2, i+.2, color='k', lw=2)
        ax.set_xticks(range(df.hand.nunique())); ax.set_xticklabels(sorted(df.hand.dropna().unique()))
        ax.set_title(name, fontsize=10); ax.grid(alpha=.3, axis='y')
    plt.tight_layout(); plt.show()

    wide = df.pivot_table(index='participant', columns='hand', values='rate_hz')
    display(wide)
    if wide.shape[1] == 2 and len(wide.dropna()) > 1:
        from scipy.stats import ttest_rel
        a, b = wide.dropna().iloc[:,0], wide.dropna().iloc[:,1]
        t_stat, p = ttest_rel(a, b)
        print(f'paired t-test on {len(a)} participants: t={t_stat:.2f}, p={p:.4f}')
        print('(with three invented participants this means nothing - it is here to copy later)')


## 4. Changing what counts as a tap

`prominence_fraction` is the setting you will actually touch: how deep a dip
has to be, as a fraction of that person's own range. Sweep it and watch how
the tap count responds. A good setting sits on a **plateau** - if the count
swings wildly with a small change, the signal is too noisy to trust.


In [ ]:
for value in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
    p = M.TapParams(prominence_fraction=value)
    t2, y2 = M.resample(t_raw, ratio, p)
    n = len(M.detect_taps(t2, M.smooth(y2, p), p)['tap_times'])
    flag = '   <- default' if value == 0.15 else ''
    print(f'prominence {value:.2f} -> {n:3d} taps{flag}')


## 5. Getting it out

`trial_metrics.csv` is one row per trial and opens in anything. Reach for
`taps.csv` when you want to model tap by tap.


In [ ]:
out = REPO / 'data' / 'processed'
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / 'from_notebook.csv', index=False)
print('written to', out / 'from_notebook.csv')


---

### Where to go next

- Change the task: `experiments/_template.js`, then `docs/CUSTOMIZE.md`
- Change what is measured: `_summarise()` in `analysis/metrics.py`
- Track the whole body instead of a hand: `tracker: 'pose'`
